# Data and model

Downloads the UCI German Credit data set (cached in `~/.cache/mlaas`), pre-processes it with MLaaS exactly as in the original Glass-Box implementation and checks it against the stored checksum, the personas and the original (shipped) decision tree.
Requires the `data` extra.

In [1]:
from glassbox import data
dh = data.load_german_credit()     # raises if the data differ from the original
print(dh.data.shape, dh.data.dtype.names)
print('checksum', data.processed_data_checksum(dh.data))

Using given data set (loaded from *mlaas.data.german_credit_simplified* module)
(1000,) ('A2', 'A5', 'A1', 'A6', 'A14', 'A3', 'A8', 'A17', 'A7', 'A20', 'A15', 'A13', 'A9', 'class')
checksum 77f23ca45c8a1154ffab954d88766f55d098810208e42466613dc780f49d6f9a


Five personas are copies of rows of the data set:

In [2]:
data.personas_matching_data(dh)

{'00': True, '02': True, '03': True, '04': True, '05': True}

In [3]:
from glassbox.model import GlassBoxModel
import numpy as np
model = GlassBoxModel()
tree = model.explainer.tree
print(tree)
print('training accuracy of the original tree:',
      np.mean(tree.predict(dh.data_onehot) == dh.target))

TreeStructure(n_nodes=387, n_leaves=194, n_features=38, classes=['1', '2'])
training accuracy of the original tree: 0.991


The original tree was trained without a `random_state`, so retraining gives a (slightly) different tree:

In [4]:
from glassbox.model import train_tree
for seed in range(3):
    new = train_tree(dh, random_state=seed)
    print(seed,
          new.leaves.shape[0],
          'leaves; agreement with the original:',
          np.mean(new.predict(dh.data_onehot) == tree.predict(dh.data_onehot)))

0 193 leaves; agreement with the original: 0.993
1 195 leaves; agreement with the original: 0.994
2 193 leaves; agreement with the original: 0.994
